# Colab-first cosmetic campaign pipeline

Run this notebook on a Google Colab GPU runtime. The maintainable implementation lives in `ai/colab_cosmetic_poster_pipeline.py`; this notebook loads that exact source so Colab and local tests use one pipeline.

Stages: EXIF normalization → PaddleOCR → rembg `isnet-general-use` with alpha matting → SDXL inpainting around the preserved product → Qwen/Ollama strict JSON → Pillow typography → three exports and ZIP.

In [ ]:
# Colab installation cell
!pip install -q --upgrade pillow rembg onnxruntime-gpu paddleocr paddlepaddle diffusers transformers accelerate safetensors torch ollama fastapi uvicorn python-multipart pyngrok

# Start Ollama separately, then make Qwen2.5 available at OLLAMA_HOST.
# The model and ngrok token are intentionally not hardcoded in this notebook.

In [ ]:
from pathlib import Path
import sys

project_root = Path('/content/Stage_1_ouvrier')
if not (project_root / 'ai').is_dir():
    project_root = Path.cwd()
ai_root = project_root / 'ai'
if not (ai_root / 'campaign_core.py').is_file():
    raise FileNotFoundError('Place the cloned project at /content/Stage_1_ouvrier or run from its root.')
sys.path.insert(0, str(ai_root))

canonical_script = ai_root / 'colab_cosmetic_poster_pipeline.py'
exec(compile(canonical_script.read_text(encoding='utf-8'), str(canonical_script), 'exec'), globals())
print(f'Loaded canonical pipeline {PIPELINE_VERSION} from {canonical_script}')

In [ ]:
# Generate one campaign from an uploaded dataset image.
from google.colab import files

uploaded = files.upload()
filename, image_bytes = next(iter(uploaded.items()))
known_brand = input('Optional brand override (press Enter for OCR): ').strip()
known_product = input('Optional product-name override (press Enter for OCR): ').strip()
known_category = input('Optional category override (press Enter for OCR): ').strip()

archive = generate_campaign_zip(
    image_bytes,
    source_name=filename,
    metadata_overrides={
        'brand': known_brand,
        'product_name': known_product,
        'category': known_category,
    },
)
output_path = Path('/content/cosmetique_ai_campaign.zip')
output_path.write_bytes(archive)
print(f'Wrote {output_path} ({len(archive):,} bytes)')
files.download(str(output_path))

In [ ]:
# Optional: expose the same pipeline to the Docker backend through ngrok.
# Set NGROK_AUTHTOKEN in the Colab runtime before running this blocking cell.
run_public_server(8000)